# L40 - Tabular Q-Learning on a Toy Queue Control Problem

**Learning objectives**
- Implement a discrete state/action control problem.
- Train a tabular Q-learning agent with epsilon-greedy exploration.
- Plot episode returns and inspect the learned policy.
- Connect the update rule to the Bellman optimality equation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class ToyQueueEnv:
    def __init__(self, max_queue=6, horizon=20):
        self.max_queue = max_queue
        self.horizon = horizon
        self.rng = np.random.default_rng(0)

    def reset(self, seed=None):
        self.rng = np.random.default_rng(seed)
        self.queue = 2
        self.t = 0
        return self.queue

    def step(self, action):
        extra_capacity = 1 if action == 1 else 0
        arrivals = int(self.rng.poisson(1.2))
        service_capacity = 1 + extra_capacity
        next_queue = min(self.max_queue, max(0, self.queue + arrivals - service_capacity))
        reward = -(next_queue + 0.5 * extra_capacity)
        self.queue = next_queue
        self.t += 1
        done = self.t >= self.horizon
        return self.queue, reward, done, {}

In [ ]:
env = ToyQueueEnv()
q_table = np.zeros((env.max_queue + 1, 2))
alpha = 0.10
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
n_episodes = 3000
returns = []

for ep in range(n_episodes):
    state = env.reset(seed=ep)
    done = False
    total_reward = 0.0
    while not done:
        if env.rng.random() < epsilon:
            action = int(env.rng.integers(0, 2))
        else:
            action = int(np.argmax(q_table[state]))

        next_state, reward, done, _ = env.step(action)
        td_target = reward + gamma * np.max(q_table[next_state]) * (not done)
        q_table[state, action] += alpha * (td_target - q_table[state, action])
        state = next_state
        total_reward += reward

    returns.append(total_reward)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

q_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
rolling = np.convolve(returns, np.ones(100) / 100, mode='valid')
axes[0].plot(rolling, color='tab:blue')
axes[0].set_title('100-episode rolling mean return')
axes[0].set_xlabel('episode')
axes[0].set_ylabel('return')
axes[0].grid(alpha=0.2)

policy = np.argmax(q_table, axis=1)
axes[1].bar(np.arange(len(policy)), policy, color='tab:orange')
axes[1].set_title('Learned policy by queue length')
axes[1].set_xlabel('queue length state')
axes[1].set_ylabel('action (0=hold, 1=add capacity)')
axes[1].set_yticks([0, 1])
fig.tight_layout()
plt.show()

policy

## Try It Yourself

1. Increase the action cost and see how the learned policy changes.
2. Increase the arrival rate from `1.2` to `1.6`.
3. Add time-of-day as part of the state and compare learning speed.